# Pamoka 13 – Agentų atmintis su Cognee žinių grafais


## Diegimas

Šiame užrašų knygelėje demonstruojama, kaip sukurti intelektualų **kodo pagalbininką** su nuolatine atmintimi naudojant [**Cognee**](https://www.cognee.ai/) žinių grafus ir **Microsoft Agent Framework** (MAF).

Cognee paverčia nestruktūruotą tekstą į struktūruotą, užklausomą žinių grafiką, paremtą vektorių įterpimais — suteikdamas jūsų agentui turtingą, santykius atitinkamą ilgalaikę atmintį.

### Ko išmoksite
1. **Kurti žinių grafus**: Paversti kūrėjų profilius ir geriausias praktikas į struktūruotas, užklausomas žinias.
2. **Integruoti Cognee su MAF**: Naudoti `@tool` funkcijas, kad MAF agentas galėtų užklausti Cognee žinių grafą.
3. **Seanso atminties pokalbiai**: Išlaikyti kontekstą per kelis klausimus tame pačiame seanse.
4. **Ilgalaikė atmintis**: Išsaugoti svarbias žinias per seansus ir jas išgauti naujuose pokalbiuose.

### Reikalavimai
- Python 3.9+
- Vietoje veikiantis Redis (`docker run -d -p 6379:6379 redis`) seansų valdymui
- LLM API raktas (pvz. OpenAI) — nustatyti `LLM_API_KEY` faile `.env`
- `CACHING=true` faile `.env` (reikalinga Cognee seansams)
- Azure AI Foundry projektas su įdiegta pokalbių modeliu
- `AZURE_AI_PROJECT_ENDPOINT` ir `AZURE_AI_MODEL_DEPLOYMENT_NAME` faile `.env`
- Azure CLI prisijungta (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Agentų atminties tipai

Šiame sąsiuvinyje nagrinėjami tie patys trys atminties tipai iš pagrindinio 13-jo pamokos sąsiuvinio, tačiau kaip ilgalaikės atminties pagrindą naudojama Cognee:

| Atminties tipas | Mechanizmas | Gyvavimo trukmė |
|---|---|---|
| **Darbinė** | `agent.create_session()` (MAF) | Viena pokalbio sesija |
| **Trumpalaikė** | Cognee sesijos talpykla (Redis) | Viena sesija |
| **Ilgalaikė** | Cognee žinių grafas + vektoriai | Nuolatinė |

### Cognee atminties architektūra
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Paruoškite Cognee saugyklą


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## 1 dalis — Žinių bazės kūrimas

Mes įtraukiame tris duomenų tipus, kad sukurtume išsamią žinių bazę mūsų programavimo asistentui:

1. **Kūrėjo profilis** — asmeninė patirtis ir techninis išsilavinimas  
2. **Geriausios Python taikymo praktikos** — Python Zen su praktiniais patarimais  
3. **Istorinės pokalbių sesijos** — ankstesni klausimų ir atsakymų seansai tarp kūrėjų ir dirbtinio intelekto asistentų


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Žinių grafiko vizualizavimas

Cognee gali sugeneruoti interaktyvią HTML vizualizaciją apie išgautus objektus ir jų tarpusavio ryšius.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Praturtinkite atmintį su Memify

`memify()` analizuoja žinių grafiką ir generuoja protingas taisykles — identifikuodama modelius, geriausias praktikas ir ryšius tarp sąvokų.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## 2 dalis — MAF agentas su Cognee įrankiais

Dabar sukursime MAF agentą, kuris gali užklausti Cognee žinių grafą naudodamas `@tool` funkcijas. Tai leidžia agentui pasinaudoti visa grafu pagrįsto semantinio paieškos galia, tuo pačiu palaikant pokalbio kontekstą per sesijas.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Darbinė atmintis su sesijomis

`AgentSession` (sukuriama per `agent.create_session()`) suteikia darbinę atmintį sesijos metu. Agentas gali vėl kreiptis į ankstesnius pranešimus, taip pat užklausti Cognee ilgalaikio žinių grafą.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nauja sesija — ilgalaikė atmintis išlieka

Pradėjus naują sesiją, darbinė atmintis išvaloma, tačiau Cognee žinių grafas lieka pasiekiamas. Agentas gali atgauti tuos pačius ilgalaikius žinių duomenis visiškai naujame pokalbyje.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Santrauka

Šiame užrašų knygelėje jūs sukūrėte kodavimo pagalbininką, kuris sujungia **MAF darbingo atmintį** (`agent.create_session()`) su **Cognee ilgalaikiu žinių grafu**.

### Ką išmokote
1. **Žinių grafų konstravimas**: Cognee apdoroja nestruktūruotą tekstą ir kuria grafiką + vektorinę atmintį.
2. **Grafų praturtinimas su memify**: Išvestos faktų ir turtingesni ryšiai virš jūsų esamo grafo.
3. **MAF + Cognee integracija**: `@tool` funkcijos leidžia MAF agentui natūraliai užklausinėti Cognee grafą.
4. **Darbingo atmintis + ilgalaikė atmintis**: `AgentSession` (per `agent.create_session()`) suteikia sesijos kontekstą, tuo tarpu Cognee teikia nuolatinę informaciją.
5. **Filtruota paieška su NodeSets**: Taikykite konkrečias žinių grafo pogrupius (pvz., tik principus).

### Pagrindinės išvados
- **Cognee** paverčia žalią tekstą į struktūruotą, santykius atpažįstančią atmintį — galingesnę nei įprasta vektorinė saugykla.
- **`@tool` funkcijos** aiškiai sujungia MAF agentus su išorinėmis žinių sistemomis.
- **`AgentSession`** (per `agent.create_session()`) palaiko pokalbio kontekstą atskirai nuo ilgalaikių žinių.
- Tas pats žinių grafas aptarnauja kelias sesijas ir agentus.

### Tikri pasaulio taikymai
- **Kūrėjų pagalbininkai**: Kodo peržiūra, incidentų analizė, architektūros pagalba
- **Klientų aptarnavimo pagalbininkai**: Pagalbos agentai, dirbantys su produktų dokumentacija, DUK ir CRM pastabomis
- **Vidaus ekspertų pagalbininkai**: Politikos, teisiniai ar saugumo asistento pagalba analizuojant gairės
- **Vieninga duomenų sluoksnis**: Sujungti struktūruotus ir nestruktūruotus duomenis į vieną užklausomą grafiką

### Tolimesni žingsniai
- Eksperimentuoti su laiko suvokimu Cognee
- Apibrėžti OWL ontologiją srities specifinės grafo kokybės nustatymui
- Pridėti naudotojų grįžtamojo ryšio kilpas, gerinančias paiešką laikui bėgant
- Plečiant kelių agentų sistemas, kurios naudoja tą patį Cognee atminties sluoksnį


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant AI vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Esant kritinei informacijai rekomenduojamas profesionalus žmogaus atliktas vertimas. Mes neatsakome už jokią painiavą ar neteisingą supratimą, kilusį dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
